<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/S4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from functools import partial
import jax
import jax.numpy as np
from flax import linen as nn
from jax.nn.initializers import lecun_normal, normal
from jax.numpy.linalg import eigh, inv, matrix_power
from jax.scipy.signal import convolve

In [2]:
if __name__=="__main__":
  rng=jax.random.PRNGKey(1)

In [3]:
rng

Array([0, 1], dtype=uint32)

In [4]:
def random_SSM(rng, N):
    a_r, b_r, c_r = jax.random.split(rng, 3)
    A = jax.random.uniform(a_r, (N, N))
    B = jax.random.uniform(b_r, (N, 1))
    C = jax.random.uniform(c_r, (1, N))
    return A, B, C

In [5]:
A,B,C=random_SSM(rng,3)
print(A,B,C)

[[0.70261526 0.23483467 0.81786454]
 [0.17094111 0.02634776 0.92260945]
 [0.6944734  0.39161062 0.7033144 ]] [[0.40364635]
 [0.548895  ]
 [0.2226392 ]] [[0.5577841  0.22405517 0.7470813 ]]


In [6]:
def discretize(A, B, C, step):
    I = np.eye(A.shape[0])
    BL = inv(I - (step / 2.0) * A)
    Ab = BL @ (I + (step / 2.0) * A)
    Bb = (BL * step) @ B
    return Ab, Bb, C

In [7]:
Ab, Bb, C=discretize(A,B,C,0.1)
print(Ab,Bb,C)

[[1.0762633  0.02616097 0.08925048]
 [0.02124082 1.004785   0.09675266]
 [0.07515424 0.041627   1.0780704 ]] [[0.04361532]
 [0.05652656]
 [0.02579223]] [[0.5577841  0.22405517 0.7470813 ]]


In [8]:
def scan_SSM(Ab, Bb, Cb, u, x0):
    def step(x_k_1, u_k):
        print(Ab.shape)
        print(x_k_1.shape)
        print(Bb.shape)
        print(u_k.shape)

        x_k = Ab @ x_k_1 + Bb @ u_k
        y_k = Cb @ x_k
        return x_k, y_k

    return jax.lax.scan(step, x0, u)

In [9]:
def run_SSM(A,B,C,u):
  L=u.shape[0]
  N=A.shape[0]
  Ab,Bb,C=discretize(A,B,C,1/L)

  return scan_SSM(Ab,Bb,C,u[:,np.newaxis],np.zeros((N,)))[1]

In [10]:
def example_mass(k,b,m):
  A=np.array([[0,1],[-k/m, -b/m]])
  B=np.array([[0],[1.0/m]])
  C=np.array([[1.0,0]])
  return A,B,C

In [11]:
def example_force(t):
  x=np.sin(10*t)
  return x*(x>0.5)

In [12]:
!pip install celluloid

In [13]:
def example_ssm():
  ssm=example_mass(k=40,b=5,m=1)
  L=100
  step=1.0/L
  ks=np.arange(L)
  u=example_force(ks*step)

  y=run_SSM(*ssm,u)


  # Plotting ---
  import matplotlib.pyplot as plt
  import seaborn
  from celluloid import Camera

  seaborn.set_context("paper")
  fig, (ax1, ax2, ax3) = plt.subplots(3)
  camera = Camera(fig)
  ax1.set_title("Force $u_k$")
  ax2.set_title("Position $y_k$")
  ax3.set_title("Object")
  ax1.set_xticks([], [])
  ax2.set_xticks([], [])

  # Animate plot over time
  for k in range(0, L, 2):
      ax1.plot(ks[:k], u[:k], color="red")
      ax2.plot(ks[:k], y[:k], color="blue")
      ax3.boxplot(
          [[y[k, 0] - 0.04, y[k, 0], y[k, 0] + 0.04]],
          showcaps=False,
          whis=False,
          vert=False,
          widths=10,
      )
      camera.snap()
  anim = camera.animate()
  anim.save("/content/sample_data/line.gif", dpi=150, writer="imagemagick")

In [14]:
if False:
    example_ssm()

In [15]:
def K_conv(Ab,Bb,Cb,L):
  return np.array(
      [(Cb@matrix_power(Ab,i)@Bb).reshape() for i in range(L)]
  )

In [16]:
# def K_conv(Ab, Bb, Cb, L):
#     kernel = []
#     for i in range(L):
#         middle = Cb @ matrix_power(Ab, i) @ Bb
#         kernel.append(middle.reshape(-1))  # reshape for flat array if needed
#     return np.array(kernel)


In [17]:
def causal_convolution(u,K,nofft=False): #1d 인과적 컨볼루션
  if nofft:
    return convolve(u,K,mode="full")[:u.shape[0]]
  else:
    assert K.shape[0]==u.shape[0] #동일 길이
    ud=np.fft.rfft(np.pad(u,(0,K.shape[0])))
    Kd=np.fft.rfft(np.pad(K,(0,u.shape[0])))
    out=ud*Kd
    return np.fft.irfft(out)[:u.shape[0]]

In [18]:
def test_cnn_is_rnn(N=4, L=16, step=1.0 / 16):
    ssm = random_SSM(rng, N)
    u = jax.random.uniform(rng, (L,))
    jax.random.split(rng, 3)
    # RNN
    rec = run_SSM(*ssm, u)

    # CNN
    ssmb = discretize(*ssm, step=step)
    conv = causal_convolution(u, K_conv(*ssmb, L))

    # Check
    assert np.allclose(rec.ravel(), conv.ravel())

In [19]:
result=test_cnn_is_rnn(4,16,1.0/16)

(4, 4)
(4,)
(4, 1)
(1,)
